In [ ]:
!python test.py --text "Hello, how are you? I hope you are doing well" --out audio/output.wav --checkpoint checkpoints/evotalk_110000.pt

In [ ]:
!pip install vocos

In [ ]:
import json
with open("data/metadata.json") as f:
    meta = json.load(f)
print(meta)

In [ ]:
from IPython.display import Image, display
display(Image("graphs/loss_total.png"))
display(Image("graphs/loss_components.png"))
display(Image("graphs/loss_pitch.png"))
display(Image("graphs/learning_rate.png"))

In [ ]:
from IPython.display import Audio
Audio("audio/output.wav")

In [ ]:
!python -c "from prepare import text_to_arpabet; print(text_to_arpabet('hello world'))"

In [ ]:
rm -rf /workspace/data

In [ ]:
rm -rf /workspace/audio

In [ ]:
rm -rf /workspace/graphs

In [ ]:
rm -rf /workspace/checkpoints

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no CUDA device visible")

In [ ]:
!nvidia-smi

In [ ]:
import os
wavs_dir = "ljspeech_raw/LJSpeech-1.1/wavs"
files = sorted(os.listdir(wavs_dir))
def avg_size(fs):
    return sum(os.path.getsize(os.path.join(wavs_dir, f)) for f in fs) / len(fs) / 1024  # KB

print("clips 0-1400:  avg KB =", avg_size(files[0:1400]))
print("clips 1400-2800: avg KB =", avg_size(files[1400:2800]))

In [ ]:
# === Copy-synthesis sanity check (no acoustic model) ===
# original  vs  (A) YOUR mel front-end -> adapter -> Vocos  vs  (B) Vocos native
import io, torch, torchaudio, numpy as np, soundfile as sf
from vocos import Vocos
import IPython.display as ipd
import prepare   # your prepare.py

device = "cuda" if torch.cuda.is_available() else "cpu"
SR = prepare.SAMPLE_RATE            # 24000

# ---- 1) grab one real speaker-9017 clip (or set AUDIO_PATH to your own file) ----
AUDIO_PATH = None
if AUDIO_PATH:
    w, sr = torchaudio.load(AUDIO_PATH)
    arr = w.mean(0).numpy()
else:
    from datasets import load_dataset, Audio
    ds = load_dataset(prepare.HF_REPO, prepare.HF_CONFIG, split=prepare.HF_SPLIT)
    ds = ds.cast_column("audio", Audio(decode=False))          # don't let datasets decode
    row = next(x for x in ds if str(x["speaker"]) == prepare.DEFAULT_SPEAKER)
    arr, sr = sf.read(io.BytesIO(row["audio"]["bytes"]))       # decode with soundfile
    arr = np.asarray(arr, np.float32)
    if arr.ndim > 1:
        arr = arr[:, 0]
    print("text:", row.get("text", ""))

wav = torch.tensor(arr, dtype=torch.float32, device=device).unsqueeze(0)
if sr != SR:
    wav = prepare.get_resampler(sr, SR)(wav)

# ---- 2) Vocos + the same mel adapter used in test_voc.py ----
vocos = Vocos.from_pretrained("charactr/vocos-mel-24khz").to(device).eval()
try:
    voc_mel_scale = vocos.feature_extractor.mel_spec.mel_scale
except AttributeError:
    voc_mel_scale = torchaudio.transforms.MelScale(
        n_mels=100, sample_rate=SR, f_min=0.0, f_max=None,
        n_stft=prepare.N_FFT // 2 + 1, norm=None, mel_scale="htk").to(device)

n_stft = prepare.N_FFT // 2 + 1
fb = torchaudio.functional.melscale_fbanks(
    n_freqs=n_stft, f_min=prepare.F_MIN, f_max=prepare.F_MAX,
    n_mels=prepare.N_MELS, sample_rate=SR, norm=None, mel_scale="htk").to(device)
fb_pinv = torch.linalg.pinv(fb.T).to(device)

@torch.no_grad()
def your_mel_to_audio(mel_log):                     # (T, n_mels) EvoTalk log-mel
    mag = torch.exp(mel_log.transpose(0, 1).contiguous())
    linear = torch.matmul(fb_pinv, mag).clamp_min(0.0)
    feats = torch.log(voc_mel_scale(linear).clamp_min(1e-7)).unsqueeze(0)
    return vocos.decode(feats).squeeze(0).cpu().numpy()

@torch.no_grad()
def vocos_native(wav):
    return vocos.decode(vocos.feature_extractor(wav.to(device))).squeeze(0).cpu().numpy()

# ---- 3) write to files and play (avoids the Audio(rate=...) quirk) ----
orig = wav.squeeze(0).cpu().numpy()
A = your_mel_to_audio(prepare.extract_mel(wav))
B = vocos_native(wav)

sf.write("orig.wav", orig, SR)
sf.write("A_adapter.wav", A, SR)
sf.write("B_native.wav", B, SR)

print("Original:")
display(ipd.Audio(filename="orig.wav"))
print("A) your mel -> adapter -> Vocos:")
display(ipd.Audio(filename="A_adapter.wav"))
print("B) Vocos native (vocoder ceiling):")
display(ipd.Audio(filename="B_native.wav"))

In [ ]:
# === Teacher-forced copy-synthesis (isolates predictors vs decoder) ===
# Compares: original clip  vs  model mel WITH ground-truth dur/pitch/energy -> adapter -> Vocos
import io, json, torch, torchaudio, numpy as np, soundfile as sf
from vocos import Vocos
import IPython.display as ipd
import prepare
from model import EvoTalk, EvoTalkConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
SR = prepare.SAMPLE_RATE
CKPT = "checkpoints/evotalk_final.pt"
META = "data/metadata.json"

meta = json.load(open(META))

# ---- load model ----
ckpt = torch.load(CKPT, map_location=device, weights_only=False)
model = EvoTalk(ckpt.get("config") or EvoTalkConfig()).to(device).eval()
model.load_state_dict(ckpt["model"])

# ---- load one val sample (has phonemes, GT durations, normalized pitch/energy, mel) ----
val_paths = [l.strip() for l in open("data/val/manifest.txt") if l.strip()]
d = torch.load(val_paths[0], weights_only=True)
pm, ps = meta["pitch_mean"], meta["pitch_std"]
em, es = meta["energy_mean"], meta["energy_std"]

phon   = d["phonemes"].unsqueeze(0).to(device)
spk    = torch.zeros(1, dtype=torch.long, device=device)
dur    = d["durations"].unsqueeze(0).to(device)
pitch  = ((torch.log1p(d["pitch"].clamp(min=0)) - pm) / ps).unsqueeze(0).to(device)   # normalized, as in training
energy = ((d["energy"] - em) / es).unsqueeze(0).to(device)
gt_mel = d["mel"].to(device)                      # (T, 80) the real target mel

# ---- Vocos + adapter (same as the copy-synth cell) ----
vocos = Vocos.from_pretrained("charactr/vocos-mel-24khz").to(device).eval()
try:
    voc_mel_scale = vocos.feature_extractor.mel_spec.mel_scale
except AttributeError:
    voc_mel_scale = torchaudio.transforms.MelScale(
        n_mels=100, sample_rate=SR, f_min=0.0, f_max=None,
        n_stft=prepare.N_FFT // 2 + 1, norm=None, mel_scale="htk").to(device)
n_stft = prepare.N_FFT // 2 + 1
fb = torchaudio.functional.melscale_fbanks(
    n_freqs=n_stft, f_min=prepare.F_MIN, f_max=prepare.F_MAX,
    n_mels=prepare.N_MELS, sample_rate=SR, norm=None, mel_scale="htk").to(device)
fb_pinv = torch.linalg.pinv(fb.T).to(device)

@torch.no_grad()
def mel_to_audio(mel_log):                         # (T, n_mels) -> np audio
    mag = torch.exp(mel_log.transpose(0, 1).contiguous())
    linear = torch.matmul(fb_pinv, mag).clamp_min(0.0)
    feats = torch.log(voc_mel_scale(linear).clamp_min(1e-7)).unsqueeze(0)
    return vocos.decode(feats).squeeze(0).cpu().numpy()

# ---- teacher-forced forward: feed GT durations + pitch + energy ----
with torch.no_grad():
    mel_out, *_ = model(
        phonemes=phon, speaker_ids=spk,
        durations=dur, pitch_targets=pitch, energy_targets=energy,
        mel_targets=None, src_mask=None, max_mel_len=gt_mel.size(0),
    )
tf_mel = mel_out.squeeze(0)                        # (T, 80)

# ---- render & play ----
sf.write("tf_gtmel.wav",   mel_to_audio(gt_mel), SR)   # real mel  -> adapter -> Vocos  (== "A")
sf.write("tf_model.wav",   mel_to_audio(tf_mel), SR)   # model mel w/ GT dur/pitch/energy
print("GT mel (teacher-forced ceiling for this clip, == your 'A'):")
display(ipd.Audio(filename="tf_gtmel.wav"))
print("MODEL mel with ground-truth durations/pitch/energy:")
display(ipd.Audio(filename="tf_model.wav"))

In [ ]:
rm -rf /workspace/A_adapter.wav

In [ ]:
rm -rf /workspace/B_native.wav

In [ ]:
rm -rf /workspace/orig.wav

In [ ]:
rm -rf /workspace/tf_gtmel.wav

In [ ]:
rm -rf /workspace/tf_model.wav

In [ ]:
# === Triangulation: front-end vs decoder vs duration predictor (native Vocos) ===
import io, json, torch, torchaudio, numpy as np, soundfile as sf
from vocos import Vocos
import IPython.display as ipd
import prepare
from model import EvoTalk, EvoTalkConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
SR = prepare.SAMPLE_RATE
meta = json.load(open("data/metadata.json"))
pm, ps = meta["pitch_mean"], meta["pitch_std"]
em, es = meta["energy_mean"], meta["energy_std"]

# --- model ---
ckpt = torch.load("checkpoints/evotalk_035000.pt", map_location=device, weights_only=False)
model = EvoTalk(ckpt.get("config") or EvoTalkConfig()).to(device).eval()
model.load_state_dict(ckpt["model"])

# --- one val utterance ---
paths = [l.strip() for l in open("data/val/manifest.txt") if l.strip()]
d = torch.load(paths[0], weights_only=True)
phon   = d["phonemes"].unsqueeze(0).to(device)
spk    = torch.tensor([int(d["speaker_id"])], device=device)
dur    = d["durations"].unsqueeze(0).to(device)
pitch  = ((torch.log1p(d["pitch"].clamp(min=0)) - pm) / ps).unsqueeze(0).to(device)
energy = ((d["energy"] - em) / es).unsqueeze(0).to(device)
gt_mel = d["mel"].to(device)                      # (T, 100) log-mel target
T = gt_mel.size(0)

# --- Vocos native ---
vocos = Vocos.from_pretrained("charactr/vocos-mel-24khz").to(device).eval()
def decode(mel_TxM):                              # (T, 100) log-mel -> np audio
    feats = mel_TxM.transpose(0, 1).unsqueeze(0).float()   # (1,100,T)
    return vocos.decode(feats).squeeze(0).cpu().numpy()

with torch.no_grad():
    # (2) teacher-forced: GT durations + GT (normalized) pitch/energy
    mel_tf = model(phon, spk, durations=dur, pitch_targets=pitch,
                   energy_targets=energy, max_mel_len=T)[0].squeeze(0)
    # (3) full inference: model predicts durations/pitch/energy itself
    mel_inf = model.inference(phon, spk).squeeze(0)

sf.write("t1_gtmel.wav",  decode(gt_mel),  SR)    # front-end ceiling (copy-synth)
sf.write("t2_teacher.wav", decode(mel_tf), SR)    # decoder, GT alignment
sf.write("t3_infer.wav",   decode(mel_inf), SR)   # full model

print(f"GT mel frames={T}   |   inference frames={mel_inf.size(0)}")
print("1) GT mel -> Vocos  (front-end ceiling; should ~= original):")
display(ipd.Audio(filename="t1_gtmel.wav"))
print("2) model mel, GT durations+pitch+energy (teacher-forced decoder):")
display(ipd.Audio(filename="t2_teacher.wav"))
print("3) full inference (model predicts everything):")
display(ipd.Audio(filename="t3_infer.wav"))

In [ ]:
# === Which predictor breaks inference? (durations vs pitch/energy) ===
with torch.no_grad():
    enc = model.encode(phon, spk, None)
    dp  = model.variance_adaptor.duration_predictor(enc)
    pred_dur = torch.clamp(torch.round(torch.exp(dp) - 1), min=1).long()

    print("GT dur sum   :", int(dur.sum().item()), " (=", T, "frames)")
    print("pred dur sum :", int(pred_dur.sum().item()), "frames")
    print("GT   durs[:25]:", dur.long().flatten()[:25].tolist())
    print("pred durs[:25]:", pred_dur.flatten()[:25].tolist())

    # Probe: GT durations (correct alignment) + PREDICTED pitch/energy
    mel_A = model(phon, spk, durations=dur,
                  pitch_targets=None, energy_targets=None, max_mel_len=T)[0].squeeze(0)

sf.write("probe_gtdur_predpe.wav", decode(mel_A), SR)
print("\nProbe = GT durations + predicted pitch/energy:")
display(ipd.Audio(filename="probe_gtdur_predpe.wav"))

In [ ]:
# === Are predicted pitch/energy out-of-range, and does "mean prosody" survive? ===
with torch.no_grad():
    xlr = model.encode(phon, spk, None)
    xlr = model.variance_adaptor.length_regulator(xlr, dur.long(), max_len=T)
    pp = model.variance_adaptor.pitch_predictor(xlr)
    ee = model.variance_adaptor.energy_predictor(xlr)
    print("pred pitch  min/mean/max: %.2f %.2f %.2f" % (pp.min(), pp.mean(), pp.max()))
    print("pred energy min/mean/max: %.2f %.2f %.2f" % (ee.min(), ee.mean(), ee.max()))
    print("(normalized GT should be ~ -3..3, mean ~0, std ~1)")

    zeros_pe = torch.zeros(1, T, device=device)
    mel_Z = model(phon, spk, durations=dur, pitch_targets=zeros_pe,
                  energy_targets=zeros_pe, max_mel_len=T)[0].squeeze(0)
sf.write("probe_zeropE.wav", decode(mel_Z), SR)
print("\nProbe = GT durations + pitch/energy forced to mean (0):")
display(ipd.Audio(filename="probe_zeropE.wav"))

In [ ]:
!cd /workspace && python3 -m http.server 8000

In [80]:
mv /workspace/checkpoints/evotalk_110000.pt /workspace/ckpt.pt